In [ ]:
"""
Huff‑based LPG Distributor Assignment (Multi‑source Dijkstra)

This script implements a fast, production‑ready Huff model assignment of LPG resellers
to inhabited 1 km pixels in Nigeria. It uses a two‑phase strategy to combine speed and accuracy:

1. Phase 1 – Approximate assignment (multi‑source Dijkstra with cell finalisation)
   A single multi‑source Dijkstra per travel mode rapidly determines the most attractive
   reseller for each pixel. Because cells are finalised once assigned, this phase introduces
   a small detour approximation but runs in ~30 seconds.

2. Phase 2 – Exact metric computation (single‑source Dijkstra per reseller)
   For each reseller, a standard Dijkstra (without finalisation) is run starting from its
   seed cell. The algorithm recomputes exact travel times and distances **only** for the
   pixels that Phase 1 assigned to that reseller. Early termination stops the search once
   all assigned pixels have been reached or a generous time limit is exceeded.

Key features:
- 8‑neighbour connectivity with symmetric edge costs (average friction × distance).
- Independent walk and motorised assignments.
- Configurable Huff parameters (β=2.0, ε=1e-6).

Outputs:
- Multi‑band GeoTIFF with car/walk shares, best reseller IDs (from Phase 1), and exact
  travel times/distances (from Phase 2).
- Lookup CSV for resellers placed on the grid.
- Optional JSON run profile.

NOTE ON THE APPROXIMATION INTRODUCED BY THE MULTI‑SOURCE DIJKSTRA
------------------------------------------------------------------
Unlike a per‑pixel Dijkstra that computes independent shortest paths to all resellers,
the multi‑source algorithm finalises a cell once it is assigned to a particular reseller.
This means other resellers cannot later “pass through” that cell, which in theory can
introduce a small detour in highly competitive areas.

We quantified this effect on a random sample of 60,000 inhabited pixels:
- Median difference in assigned travel time: 1.98 min
- 95th percentile difference: 23.30 min
- Maximum observed difference: 272.43 min
- Pixels with deviation > 1 min: 57.2%
- Pixels with deviation > 5 min: 36.3%
- Pixels with deviation > 10 min: 19.4%

The two‑phase design eliminates this approximation from the final travel metrics.
Phase 1 still provides the fast assignment, while Phase 2 supplies exact times and distances.
The overall runtime increases moderately (typically under 2 minutes) while retaining the
massive speedup over a full per‑pixel Huff model.


**Sensitivity of Huff Assignment to Attractiveness Transformation**

A sensitivity analysis was conducted to assess how the Huff‑based distributor assignment responds 
to different transformations of the original attractiveness scores \(A \in [0,1]\). 
Six transformations were tested: the baseline \(A\), multiplicative scalings \(A \times 5\) and \(A \times 100\), 
and additive shifts \((A+1)\), \((A+2)\), together with their scaled variants \(5 \times (A+1)\) and \(5 \times (A+2)\). 
For each case, the multi‑source Dijkstra algorithm was re‑run for both walk and motorised modes, and the resulting 
assignments were compared.

**Key findings:**  
- **Multiplicative scaling leaves the results unchanged.** All pure scaling transformations produced identical 
assigned reseller IDs and identical median travel times/distances. This confirms that the Huff weight \(\sqrt{A}\) 
is insensitive to a uniform rescaling of attractiveness; only relative differences matter.  
- **Adding a constant alters the spatial allocation.** Introducing an additive term (e.g., \(A+1\) or \(A+2\)) raises 
the minimum attractiveness and compresses the relative variation among resellers. This leads to a systematic, though 
modest, reduction in median travel times (e.g., walk time from 368 to ~353 minutes) and distances, and changes the 
assigned reseller for about 14–17% of inhabited pixels compared to the baseline.  
- **The effect is governed by the additive constant, not by an outer scaling factor.** The results for 
\((A+1)\) and \(5 \times (A+1)\) are identical, as are those for \((A+2)\) and \(5 \times (A+2)\). The magnitude 
of the shift (i.e., the value of the constant) controls the degree of departure from the baseline allocation.

**Implications for model specification:**  
Given the invariance to multiplicative scaling, the original 0–1 range of attractiveness is retained for simplicity 
and interpretability. If, for policy or behavioural reasons, one wishes to **increase the relative importance of 
distance over attractiveness**, adding a constant to \(A\) (e.g., using \(\sqrt{A+c}\) instead of \(\sqrt{A}\)) 
provides a practical and computationally efficient solution. This approach preserves the additive cost structure 
of the Dijkstra algorithm and avoids the complications that would arise from raising travel time to a 
power \(\beta \neq 1\), which would require non‑additive edge weights and potentially more complex path‑finding 
routines. In essence, adding a constant limits the relative differences in attractiveness and thus gives greater 
weight to proximity, achieving an effect analogous to increasing the distance exponent in a classical Huff model, 
but without altering the core algorithmic framework.


Attractiveness expansion: how does it works?

The algorithm runs a multi‑source Dijkstra search where each reseller expands simultaneously. It uses a priority 
queue that always contains the border pixels that have been reached but not yet assigned. Every queue entry stores 
a key (a cost) equal to travel_time / sqrt(attractiveness). At each step, the algorithm extracts the pixel with 
the smallest key (low cost), finalizes it (assigns it to that reseller), and then expands from it to its 8 
neighbors. Any newly reached neighbor is inserted into the queue with its own key. If a neighbor was already 
in the queue with a larger key, the smaller key takes precedence. 
This process continues until all inhabited pixels are assigned. Crucially, the physical expansion speed 
(pixels per real minute) depends only on the friction surface and distance, not on attractiveness. 
Attractiveness does not make a reseller’s wave move faster in real time; it only reorders the queue: a 
higher sqrt(attractiveness) lowers the key, giving that reseller priority over others at equal travel time. 
Therefore the order of assignment is driven by minimizing travel_time / sqrt(attractiveness), 
not by any “expansion velocity” proportional to that ratio. In competition, a more attractive reseller will 
“jump the queue” and claim pixels earlier, but its actual arrival time at each pixel is exactly the same 
as it would be without attractiveness.





"""

from __future__ import annotations

import heapq
import json
import math
import time
from pathlib import Path
from typing import Dict, List, Sequence, Tuple, Optional

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import rowcol
from rasterio.warp import Resampling, reproject

# -----------------------------------------------------------------------------
# USER CONFIGURATION – adjust these paths and parameters as needed
# -----------------------------------------------------------------------------
DATASET_DIR = Path("dataset_big")

# Input files (relative to DATASET_DIR)
POPULATION_TIF = "Population.tif"
CAR_SHARE_TIF = "vehicles_allocation_share.tif"
FRICTION_WALK_TIF = "friction_walk.tif"
FRICTION_MOTO_TIF = "friction_moto.tif"
RESELLER_GPKG = "full_lpg_chain_nig_3857.gpkg"
RESELLER_LAYER = "resell_and_filling"

# Column names in the reseller layer
RESELLER_ID_COL = "id_res&fil"
ATTRACTIVENESS_COL = "attractiveness"

# Output files (saved inside DATASET_DIR)
OUTPUT_RASTER = "huff_preferred_distributor_per_pixel.tif"
OUTPUT_LOOKUP = "huff_reseller_lookup.csv"
OUTPUT_PROFILE = "huff_run_profile.json"          # set to None to disable

# Huff model parameters
BETA = 2.0
EPS = 1e-6
ATTR_MIN = 1e-6

# Raster nodata value
NODATA = -9999.0

# Progress reporting frequency (pixels)
LOG_EVERY = 100000

# Pixel size in metres (grid is EPSG:3857, 1 km cells)
PIXEL_SIZE_M = 1000.0

# Factor for early termination in exact recomputation (multiply max observed time from Phase 1)
TIME_LIMIT_FACTOR = 1.5

# -----------------------------------------------------------------------------
# 8‑neighbour moves with Euclidean distance factors (in pixel units)
# -----------------------------------------------------------------------------
NEIGHBORS = (
    (-1,  0, 1.0),
    ( 1,  0, 1.0),
    ( 0, -1, 1.0),
    ( 0,  1, 1.0),
    (-1, -1, math.sqrt(2.0)),
    (-1,  1, math.sqrt(2.0)),
    ( 1, -1, math.sqrt(2.0)),
    ( 1,  1, math.sqrt(2.0)),
)

# -----------------------------------------------------------------------------
# Helper functions
# -----------------------------------------------------------------------------
def _safe_ratio(num: float, den: float) -> float:
    """Return num/den, or 0.0 if den == 0."""
    return float(num / den) if den else 0.0


def _print_progress(
    mode: str,
    assigned: int,
    total: int,
    elapsed: float,
) -> None:
    """Print progress with speed and ETA."""
    if total <= 0:
        return
    pct = 100.0 * assigned / total
    rate = _safe_ratio(assigned, elapsed)
    remaining = max(total - assigned, 0)
    eta_sec = _safe_ratio(remaining, max(rate, 1e-12))
    print(
        f"[{mode}] {assigned:,}/{total:,} ({pct:.2f}%) | "
        f"{rate:.1f} px/s | ETA {eta_sec/60:.1f} min"
    )


def read_reference_population(path: Path) -> Tuple[np.ndarray, Dict]:
    """Read population raster and return array + profile."""
    with rasterio.open(path) as src:
        pop = src.read(1).astype(np.float32)
        profile = src.profile.copy()
    return pop, profile


def reproject_to_reference(
    src_path: Path,
    ref_profile: Dict,
    resampling: Resampling,
) -> np.ndarray:
    """Reproject (or resample) a raster to match the reference grid."""
    dst = np.full(
        (ref_profile["height"], ref_profile["width"]),
        np.nan,
        dtype=np.float32,
    )
    with rasterio.open(src_path) as src:
        reproject(
            source=rasterio.band(src, 1),
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src.nodata,
            dst_transform=ref_profile["transform"],
            dst_crs=ref_profile["crs"],
            dst_nodata=np.nan,
            resampling=resampling,
        )
    return dst


def normalize_car_share(arr: np.ndarray) -> np.ndarray:
    """Convert car share to fraction (0–1). Assumes input may be 0–100."""
    out = arr.astype(np.float32, copy=True)
    finite = np.isfinite(out)
    if finite.any():
        vmax = float(np.nanmax(out[finite]))
        if vmax > 1.0 + 1e-4 and vmax <= 100.0 + 1e-4:
            out[finite] = out[finite] / 100.0
    out[~finite] = 0.0
    np.clip(out, 0.0, 1.0, out=out)
    return out


def sanitize_friction(arr: np.ndarray) -> np.ndarray:
    """Set non‑positive or non‑finite friction to NaN."""
    out = arr.astype(np.float32, copy=True)
    invalid = ~np.isfinite(out) | (out <= 0.0)
    out[invalid] = np.nan
    return out


def connected_components_8(mask: np.ndarray) -> np.ndarray:
    """
    Label 8‑connected components of True cells.
    Returns an integer array (0 = background).
    Falls back to pure Python if scipy is not available.
    """
    # Try to use scipy first (much faster)
    try:
        from scipy import ndimage
        structure = np.ones((3, 3), dtype=np.uint8)
        labels, _ = ndimage.label(mask, structure=structure)
        return labels.astype(np.int32)
    except ImportError:
        # Fallback pure Python implementation
        h, w = mask.shape
        labels = np.zeros((h, w), dtype=np.int32)
        current = 0
        stack = []
        for r in range(h):
            for c in range(w):
                if not mask[r, c] or labels[r, c] != 0:
                    continue
                current += 1
                labels[r, c] = current
                stack.append((r, c))
                while stack:
                    rr, cc = stack.pop()
                    for dr, dc, _ in NEIGHBORS:
                        nr, nc = rr + dr, cc + dc
                        if nr < 0 or nr >= h or nc < 0 or nc >= w:
                            continue
                        if not mask[nr, nc] or labels[nr, nc] != 0:
                            continue
                        labels[nr, nc] = current
                        stack.append((nr, nc))
        return labels


def load_reseller_seeds(
    gpkg_path: Path,
    layer: str,
    id_col: str,
    attr_col: str,
    ref_profile: Dict,
) -> List[Dict]:
    """
    Load reseller points, reproject to reference CRS, and keep only those
    falling inside the raster extent. If multiple points map to the same pixel,
    keep the most attractive one.
    Returns a list of dicts with keys: reseller_id, attractiveness, row, col.
    """
    gdf = gpd.read_file(gpkg_path, layer=layer)
    if gdf.empty:
        raise ValueError("Reseller layer is empty.")

    ref_crs = ref_profile["crs"]
    if gdf.crs is None:
        raise ValueError("Reseller GeoPackage has no CRS.")
    if gdf.crs != ref_crs:
        gdf = gdf.to_crs(ref_crs)

    # Clean IDs and attractiveness
    ids = pd.to_numeric(gdf[id_col], errors="coerce")
    attrs = pd.to_numeric(gdf[attr_col], errors="coerce")
    attrs = attrs.fillna(ATTR_MIN).clip(lower=ATTR_MIN)

    xs = gdf.geometry.x.to_numpy()
    ys = gdf.geometry.y.to_numpy()
    rows, cols = rowcol(ref_profile["transform"], xs, ys)

    h, w = ref_profile["height"], ref_profile["width"]
    seeds_dict = {}
    for rid, attr, r, c in zip(ids, attrs, rows, cols):
        if not np.isfinite(rid):
            continue
        rr = int(r)
        cc = int(c)
        if rr < 0 or rr >= h or cc < 0 or cc >= w:
            continue

        key = (rr, cc)
        candidate = {
            "reseller_id": float(rid),
            "attractiveness": float(attr),
            "row": rr,
            "col": cc,
        }
        prev = seeds_dict.get(key)
        if prev is None or candidate["attractiveness"] > prev["attractiveness"]:
            seeds_dict[key] = candidate

    seeds = list(seeds_dict.values())
    if not seeds:
        raise ValueError("No reseller seed mapped onto the reference grid.")
    return seeds


def run_multisource_dijkstra(
    mode_name: str,
    friction: np.ndarray,
    traversable: np.ndarray,
    target_mask: np.ndarray,
    labels: np.ndarray,
    seeds: List[Dict],
    pixel_size_m: float,
    log_every: int = LOG_EVERY,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, int, float]:
    """
    Multi‑source Dijkstra con Huff‑style scoring (β=2).

    Returns:
        best_seed_idx : 2D array of seed indices (or -1)
        best_time_min : 2D array of travel times in minutes (or inf)
        best_dist_km  : 2D array of path distance in km (or inf)
        assigned      : number of target pixels assigned
        elapsed_sec   : solver wall time
    """
    h, w = friction.shape
    n_seeds = len(seeds)

    # Precompute source data
    seed_rows = np.array([s["row"] for s in seeds], dtype=np.int32)
    seed_cols = np.array([s["col"] for s in seeds], dtype=np.int32)
    sqrt_attr = np.array(
        [math.sqrt(max(s["attractiveness"], ATTR_MIN)) for s in seeds],
        dtype=np.float64,
    )

    # Determine which connected components contain at least one seed
    seed_labels = labels[seed_rows, seed_cols]
    max_label = labels.max()
    has_seed = np.zeros(max_label + 1, dtype=bool)
    has_seed[seed_labels[seed_labels > 0]] = True

    # State arrays
    best_adj = np.full((h, w), np.inf, dtype=np.float64)   # adjusted time
    best_time = np.full((h, w), np.inf, dtype=np.float64)  # raw time (minutes)
    best_dist = np.full((h, w), np.inf, dtype=np.float64)  # distance (km)
    best_seed = np.full((h, w), -1, dtype=np.int32)
    finalized = np.zeros((h, w), dtype=bool)

    # Priority queue: (adj_time, row, col, seed_idx)
    heap = []
    for si in range(n_seeds):
        r = seed_rows[si]
        c = seed_cols[si]
        if not traversable[r, c]:
            continue
        # At source pixel, tie‑break in favour of higher attractiveness
        if (0.0 < best_adj[r, c]) or (
            best_adj[r, c] == 0.0
            and best_seed[r, c] >= 0
            and sqrt_attr[si] > sqrt_attr[best_seed[r, c]]
        ):
            best_adj[r, c] = 0.0
            best_time[r, c] = 0.0
            best_dist[r, c] = 0.0
            best_seed[r, c] = si
            heapq.heappush(heap, (0.0, r, c, si))

    total_targets = int(np.count_nonzero(target_mask & traversable))
    assigned = 0
    t_start = time.perf_counter()
    last_log = 0

    while heap and assigned < total_targets:
        adj, r, c, si = heapq.heappop(heap)
        if finalized[r, c]:
            continue
        if adj > best_adj[r, c]:
            continue

        finalized[r, c] = True
        if target_mask[r, c]:
            assigned += 1
            if assigned - last_log >= log_every:
                _print_progress(
                    mode_name,
                    assigned,
                    total_targets,
                    time.perf_counter() - t_start,
                )
                last_log = assigned

        inv_scale = 1.0 / sqrt_attr[si]
        t_curr = best_time[r, c]
        d_curr = best_dist[r, c]

        for dr, dc, dist_factor in NEIGHBORS:
            nr = r + dr
            nc = c + dc
            if nr < 0 or nr >= h or nc < 0 or nc >= w:
                continue
            if finalized[nr, nc] or not traversable[nr, nc]:
                continue

            # Edge cost: average friction × distance (metres) → minutes
            edge_time = float(
                0.5 * (friction[r, c] + friction[nr, nc])
                * (pixel_size_m * dist_factor)
            )
            t_new = t_curr + edge_time
            adj_new = t_new * inv_scale

            # Distance in km (geometric, independent of friction)
            edge_km = (pixel_size_m * dist_factor) / 1000.0
            d_new = d_curr + edge_km

            # Lexicographic tie‑break: lower adjusted time, then lower raw time
            if (adj_new + 1e-12) < best_adj[nr, nc] or (
                abs(adj_new - best_adj[nr, nc]) <= 1e-12
                and t_new < best_time[nr, nc]
            ):
                best_adj[nr, nc] = adj_new
                best_time[nr, nc] = t_new
                best_dist[nr, nc] = d_new
                best_seed[nr, nc] = si
                heapq.heappush(heap, (adj_new, nr, nc, si))

    # -------------------------------------------------------------------------
    # Component‑aware fallback
    # -------------------------------------------------------------------------
    unassigned = target_mask & traversable & (best_seed < 0)
    eligible = unassigned & has_seed[labels]
    fallback_hits = 0
    if np.any(eligible):
        unique_lbls = np.unique(labels[eligible])
        for lbl in unique_lbls:
            if lbl <= 0:
                continue
            comp_mask = labels == lbl
            comp_targets = eligible & comp_mask
            if not np.any(comp_targets):
                continue

            comp_seed_idx = np.where(seed_labels == lbl)[0]
            if comp_seed_idx.size == 0:
                continue

            fallback_hits += int(np.count_nonzero(comp_targets))

            # Reset state inside the component
            best_adj_comp = np.full((h, w), np.inf, dtype=np.float64)
            best_time_comp = np.full((h, w), np.inf, dtype=np.float64)
            best_dist_comp = np.full((h, w), np.inf, dtype=np.float64)
            best_seed_comp = np.full((h, w), -1, dtype=np.int32)
            finalized_comp = np.zeros((h, w), dtype=bool)

            heap_comp = []
            for si in comp_seed_idx:
                r0 = seed_rows[si]
                c0 = seed_cols[si]
                if not comp_mask[r0, c0] or not traversable[r0, c0]:
                    continue
                best_adj_comp[r0, c0] = 0.0
                best_time_comp[r0, c0] = 0.0
                best_dist_comp[r0, c0] = 0.0
                best_seed_comp[r0, c0] = si
                heapq.heappush(heap_comp, (0.0, r0, c0, si))

            while heap_comp:
                adj, r, c, si = heapq.heappop(heap_comp)
                if finalized_comp[r, c]:
                    continue
                if adj > best_adj_comp[r, c]:
                    continue
                if not comp_mask[r, c]:
                    continue

                finalized_comp[r, c] = True

                inv_scale = 1.0 / sqrt_attr[si]
                t_curr = best_time_comp[r, c]
                d_curr = best_dist_comp[r, c]

                for dr, dc, dist_factor in NEIGHBORS:
                    nr = r + dr
                    nc = c + dc
                    if nr < 0 or nr >= h or nc < 0 or nc >= w:
                        continue
                    if (
                        finalized_comp[nr, nc]
                        or not traversable[nr, nc]
                        or not comp_mask[nr, nc]
                    ):
                        continue

                    edge_time = float(
                        0.5 * (friction[r, c] + friction[nr, nc])
                        * (pixel_size_m * dist_factor)
                    )
                    t_new = t_curr + edge_time
                    adj_new = t_new * inv_scale

                    edge_km = (pixel_size_m * dist_factor) / 1000.0
                    d_new = d_curr + edge_km

                    if (adj_new + 1e-12) < best_adj_comp[nr, nc] or (
                        abs(adj_new - best_adj_comp[nr, nc]) <= 1e-12
                        and t_new < best_time_comp[nr, nc]
                    ):
                        best_adj_comp[nr, nc] = adj_new
                        best_time_comp[nr, nc] = t_new
                        best_dist_comp[nr, nc] = d_new
                        best_seed_comp[nr, nc] = si
                        heapq.heappush(heap_comp, (adj_new, nr, nc, si))

            # Merge fallback results into global arrays
            update = comp_targets & (best_seed_comp >= 0)
            best_seed[update] = best_seed_comp[update]
            best_time[update] = best_time_comp[update]
            best_dist[update] = best_dist_comp[update]

    elapsed = time.perf_counter() - t_start
    assigned_final = int(np.count_nonzero(target_mask & traversable & (best_seed >= 0)))
    _print_progress(mode_name, assigned_final, total_targets, elapsed)

    return best_seed, best_time, best_dist, assigned_final, elapsed


def seeds_to_raster(
    best_seed_idx: np.ndarray,
    best_time_min: np.ndarray,
    best_dist_km: np.ndarray,
    seeds: List[Dict],
    inhabited_mask: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Convert seed indices, times and distances to final rasters."""
    h, w = best_seed_idx.shape
    id_raster = np.full((h, w), NODATA, dtype=np.float32)
    time_raster = np.full((h, w), NODATA, dtype=np.float32)
    dist_raster = np.full((h, w), NODATA, dtype=np.float32)

    valid = inhabited_mask & (best_seed_idx >= 0) & np.isfinite(best_time_min)
    if np.any(valid):
        seed_ids = np.array([s["reseller_id"] for s in seeds], dtype=np.float64)
        idx = best_seed_idx[valid]
        id_raster[valid] = seed_ids[idx].astype(np.float32)
        time_raster[valid] = best_time_min[valid].astype(np.float32)
        dist_raster[valid] = best_dist_km[valid].astype(np.float32)

    return id_raster, time_raster, dist_raster


def write_multiband_raster(
    out_path: Path,
    ref_profile: Dict,
    bands: Sequence[np.ndarray],
    band_names: Sequence[str],
) -> None:
    """Write a multi‑band GeoTIFF with LZW compression and band descriptions."""
    profile = ref_profile.copy()
    profile.update(
        driver="GTiff",
        dtype="float32",
        count=len(bands),
        compress="lzw",
        nodata=NODATA,
        tiled=True,
        bigtiff="IF_SAFER",
    )

    # Ensure tile sizes are multiples of 16
    h, w = profile["height"], profile["width"]
    tile_x = min(256, w)
    tile_y = min(256, h)
    tile_x = max(16, (tile_x // 16) * 16)
    tile_y = max(16, (tile_y // 16) * 16)
    profile["blockxsize"] = tile_x
    profile["blockysize"] = tile_y

    with rasterio.open(out_path, "w", **profile) as dst:
        for i, (arr, name) in enumerate(zip(bands, band_names), start=1):
            dst.write(arr.astype(np.float32), i)
            dst.set_band_description(i, name)


# -----------------------------------------------------------------------------
# Exact metric recomputation (Phase 2)
# -----------------------------------------------------------------------------
def recompute_exact_metrics(
    mode_name: str,
    friction: np.ndarray,
    traversable: np.ndarray,
    id_raster: np.ndarray,                # Phase‑1 assignment (IDs, NODATA elsewhere)
    phase1_time_raster: np.ndarray,       # approximate times from Phase 1 (for limit)
    seeds: List[Dict],
    pixel_size_m: float,
    time_limit_factor: float = TIME_LIMIT_FACTOR,
    log_every: int = 500,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Recompute exact travel times and distances for pixels assigned to each reseller.
    For every seed, run a single‑source Dijkstra (no finalisation) and record
    the true metrics only for pixels whose Phase‑1 assignment matches the seed.

    Early termination:
        - Stop when all assigned pixels of the current seed have been reached.
        - Alternatively, stop when the time exceeds `max_phase1_time * time_limit_factor`.
          If this second condition triggers, a warning is printed.

    Returns:
        exact_time_raster : 2D array with exact travel times (minutes, NODATA for unassigned)
        exact_dist_raster : 2D array with exact distances (km, NODATA for unassigned)
    """
    h, w = friction.shape
    exact_time = np.full((h, w), NODATA, dtype=np.float32)
    exact_dist = np.full((h, w), NODATA, dtype=np.float32)

    # Build mapping from seed ID to its index
    seed_id_to_idx = {}
    for si, s in enumerate(seeds):
        sid = s["reseller_id"]
        if sid not in seed_id_to_idx:
            seed_id_to_idx[sid] = si

    assigned_mask = (id_raster != NODATA) & traversable
    unique_ids = np.unique(id_raster[assigned_mask])

    n_seeds_total = len(unique_ids)
    n_processed = 0
    t_start = time.perf_counter()
    last_log = 0
    warnings_issued = set()

    print(f"[{mode_name}-exact] Processing {n_seeds_total:,} resellers...")

    for sid in unique_ids:
        si = seed_id_to_idx.get(sid)
        if si is None:
            continue
        seed = seeds[si]
        sr, sc = seed["row"], seed["col"]
        if not traversable[sr, sc]:
            continue

        target_mask = assigned_mask & (id_raster == sid)
        target_coords = np.argwhere(target_mask)  # list of (r,c)
        target_count = len(target_coords)
        if target_count == 0:
            continue

        # Determine maximum allowed time for early termination
        phase1_times = phase1_time_raster[target_mask]
        max_phase1_time = np.max(phase1_times) if len(phase1_times) > 0 else 0.0
        time_limit = max_phase1_time * time_limit_factor if max_phase1_time > 0 else np.inf

        # Dijkstra state
        dist_time = np.full((h, w), np.inf, dtype=np.float64)
        dist_dist = np.full((h, w), np.inf, dtype=np.float64)
        visited = np.zeros((h, w), dtype=bool)

        dist_time[sr, sc] = 0.0
        dist_dist[sr, sc] = 0.0
        heap = [(0.0, sr, sc)]

        reached_targets = 0
        target_set = set((r, c) for r, c in target_coords)

        while heap:
            t, r, c = heapq.heappop(heap)
            if visited[r, c]:
                continue
            visited[r, c] = True

            # If time limit exceeded (and we haven't reached all targets), stop early
            if t > time_limit and reached_targets < target_count:
                if sid not in warnings_issued:
                    print(f"  Warning: Seed {sid} exceeded time limit ({t:.1f} > {time_limit:.1f} min) "
                          f"before reaching all targets ({reached_targets}/{target_count}).")
                    warnings_issued.add(sid)
                break

            # If this cell is a target for this seed, record exact metrics
            if (r, c) in target_set:
                exact_time[r, c] = t
                exact_dist[r, c] = dist_dist[r, c]
                reached_targets += 1
                if reached_targets == target_count:
                    break  # all done

            t_curr = t
            d_curr = dist_dist[r, c]

            for dr, dc, dist_factor in NEIGHBORS:
                nr = r + dr
                nc = c + dc
                if nr < 0 or nr >= h or nc < 0 or nc >= w:
                    continue
                if visited[nr, nc] or not traversable[nr, nc]:
                    continue

                edge_time = float(
                    0.5 * (friction[r, c] + friction[nr, nc])
                    * (pixel_size_m * dist_factor)
                )
                t_new = t_curr + edge_time
                edge_km = (pixel_size_m * dist_factor) / 1000.0
                d_new = d_curr + edge_km

                if t_new < dist_time[nr, nc] - 1e-12:
                    dist_time[nr, nc] = t_new
                    dist_dist[nr, nc] = d_new
                    heapq.heappush(heap, (t_new, nr, nc))

        # End while heap

        n_processed += 1
        if n_processed - last_log >= log_every:
            elapsed = time.perf_counter() - t_start
            pct = 100.0 * n_processed / n_seeds_total
            rate = _safe_ratio(n_processed, elapsed)
            remaining = n_seeds_total - n_processed
            eta = _safe_ratio(remaining, max(rate, 1e-12))
            print(
                f"[{mode_name}-exact] {n_processed:,}/{n_seeds_total:,} resellers done "
                f"({pct:.1f}%) | {rate:.1f} res/s | ETA {eta/60:.1f} min"
            )
            last_log = n_processed

    # End loop over seeds
    elapsed = time.perf_counter() - t_start
    print(f"[{mode_name}-exact] All resellers processed. Assigned pixels recomputed.")

    return exact_time, exact_dist


# -----------------------------------------------------------------------------
# Main pipeline
# -----------------------------------------------------------------------------
def run_huff_pipeline(dataset_dir: Path = DATASET_DIR) -> Dict:
    """
    Execute the full Huff assignment and write outputs.

    Returns a dictionary with run statistics.
    """
    ds = Path(dataset_dir)
    t_total_start = time.perf_counter()

    # -------------------------------------------------------------------------
    # Load and prepare rasters
    # -------------------------------------------------------------------------
    t0 = time.perf_counter()
    pop, ref_profile = read_reference_population(ds / POPULATION_TIF)
    inhabited = np.isfinite(pop) & (pop > 0.0)
    t_read_pop = time.perf_counter() - t0

    t0 = time.perf_counter()
    car_share_raw = reproject_to_reference(
        ds / CAR_SHARE_TIF, ref_profile, Resampling.bilinear
    )
    walk_friction = reproject_to_reference(
        ds / FRICTION_WALK_TIF, ref_profile, Resampling.bilinear
    )
    moto_friction = reproject_to_reference(
        ds / FRICTION_MOTO_TIF, ref_profile, Resampling.bilinear
    )

    car_share = normalize_car_share(car_share_raw)
    walk_share = 1.0 - car_share

    walk_friction = sanitize_friction(walk_friction)
    moto_friction = sanitize_friction(moto_friction)
    t_reproject = time.perf_counter() - t0

    # -------------------------------------------------------------------------
    # Load reseller seeds
    # -------------------------------------------------------------------------
    t0 = time.perf_counter()
    seeds_all = load_reseller_seeds(
        gpkg_path=ds / RESELLER_GPKG,
        layer=RESELLER_LAYER,
        id_col=RESELLER_ID_COL,
        attr_col=ATTRACTIVENESS_COL,
        ref_profile=ref_profile,
    )
    t_load_seeds = time.perf_counter() - t0

    # Build traversable masks
    walk_traversable = np.isfinite(walk_friction) & (walk_friction > 0.0)
    moto_traversable = np.isfinite(moto_friction) & (moto_friction > 0.0)

    seeds_walk = [s for s in seeds_all if walk_traversable[s["row"], s["col"]]]
    seeds_moto = [s for s in seeds_all if moto_traversable[s["row"], s["col"]]]

    if not seeds_walk:
        raise RuntimeError("No walkable reseller seeds.")
    if not seeds_moto:
        raise RuntimeError("No motorised‑accessible reseller seeds.")

    # Save lookup CSV (unique seeds per pixel, merging both modes)
    lookup_dict = {}
    for s in seeds_walk + seeds_moto:
        key = (s["row"], s["col"])
        lookup_dict[key] = s
    lookup_seeds = list(lookup_dict.values())
    pd.DataFrame(lookup_seeds).to_csv(ds / OUTPUT_LOOKUP, index=False)

    # -------------------------------------------------------------------------
    # Connected components
    # -------------------------------------------------------------------------
    t0 = time.perf_counter()
    walk_labels = connected_components_8(walk_traversable)
    moto_labels = connected_components_8(moto_traversable)
    t_components = time.perf_counter() - t0

    print(
        f"Inhabited pixels: {inhabited.sum():,} | "
        f"Walk seeds: {len(seeds_walk):,} | Moto seeds: {len(seeds_moto):,}"
    )

    # -------------------------------------------------------------------------
    # Phase 1: Approximate assignment (multi‑source Dijkstra)
    # -------------------------------------------------------------------------
    walk_seed_idx, walk_time_approx, walk_dist_approx, walk_assigned, t_walk_solver = run_multisource_dijkstra(
        mode_name="walk",
        friction=walk_friction,
        traversable=walk_traversable,
        target_mask=inhabited,
        labels=walk_labels,
        seeds=seeds_walk,
        pixel_size_m=PIXEL_SIZE_M,
    )

    moto_seed_idx, moto_time_approx, moto_dist_approx, moto_assigned, t_moto_solver = run_multisource_dijkstra(
        mode_name="moto",
        friction=moto_friction,
        traversable=moto_traversable,
        target_mask=inhabited,
        labels=moto_labels,
        seeds=seeds_moto,
        pixel_size_m=PIXEL_SIZE_M,
    )

    # Convert to ID rasters (using approximate times/distances for now)
    walk_id, walk_time_approx_out, walk_dist_approx_out = seeds_to_raster(
        walk_seed_idx, walk_time_approx, walk_dist_approx, seeds_walk, inhabited
    )
    moto_id, moto_time_approx_out, moto_dist_approx_out = seeds_to_raster(
        moto_seed_idx, moto_time_approx, moto_dist_approx, seeds_moto, inhabited
    )

    # -------------------------------------------------------------------------
    # Phase 2: Exact metric recomputation
    # -------------------------------------------------------------------------
    t0_exact = time.perf_counter()
    walk_time_exact, walk_dist_exact = recompute_exact_metrics(
        mode_name="walk",
        friction=walk_friction,
        traversable=walk_traversable,
        id_raster=walk_id,
        phase1_time_raster=walk_time_approx_out,
        seeds=seeds_walk,
        pixel_size_m=PIXEL_SIZE_M,
    )
    moto_time_exact, moto_dist_exact = recompute_exact_metrics(
        mode_name="moto",
        friction=moto_friction,
        traversable=moto_traversable,
        id_raster=moto_id,
        phase1_time_raster=moto_time_approx_out,
        seeds=seeds_moto,
        pixel_size_m=PIXEL_SIZE_M,
    )
    t_exact = time.perf_counter() - t0_exact








    # ---------------------------------------------------------------------
    # Optional diagnostic: compare approximate vs exact metrics
    # ---------------------------------------------------------------------
    def _print_comparison(mode, approx_time, exact_time, approx_dist, exact_dist, mask):
        valid = mask & (approx_time != NODATA) & (exact_time != NODATA)
        if np.any(valid):
            time_diff = np.abs(approx_time[valid] - exact_time[valid])
            dist_diff = np.abs(approx_dist[valid] - exact_dist[valid])
            print(f"[{mode}] Approx vs exact time diff (min): median={np.median(time_diff):.2f}, max={np.max(time_diff):.2f}")
            print(f"[{mode}] Approx vs exact dist diff (km):  median={np.median(dist_diff):.2f}, max={np.max(dist_diff):.2f}")

    _print_comparison("walk", walk_time_approx_out, walk_time_exact, walk_dist_approx_out, walk_dist_exact, inhabited)
    _print_comparison("moto", moto_time_approx_out, moto_time_exact, moto_dist_approx_out, moto_dist_exact, inhabited)










    # Replace approximate time/distance rasters with exact ones
    walk_time_out = walk_time_exact
    walk_dist_out = walk_dist_exact
    moto_time_out = moto_time_exact
    moto_dist_out = moto_dist_exact

    # -------------------------------------------------------------------------
    # Prepare share bands with nodata outside inhabited areas
    # -------------------------------------------------------------------------
    car_band = np.full_like(car_share, NODATA, dtype=np.float32)
    walk_band = np.full_like(walk_share, NODATA, dtype=np.float32)
    car_band[inhabited] = car_share[inhabited]
    walk_band[inhabited] = walk_share[inhabited]

    # -------------------------------------------------------------------------
    # Write multi‑band GeoTIFF
    # -------------------------------------------------------------------------
    write_multiband_raster(
        ds / OUTPUT_RASTER,
        ref_profile,
        bands=(
            car_band,
            walk_band,
            walk_id,
            walk_time_out,
            walk_dist_out,
            moto_id,
            moto_time_out,
            moto_dist_out,
        ),
        band_names=[
            "car_share",
            "walk_share",
            "best_reseller_id_walk",
            "best_time_walk_min",
            "best_distance_walk_km",
            "best_reseller_id_car",
            "best_time_car_min",
            "best_distance_car_km",
        ],
    )

    # -------------------------------------------------------------------------
    # Final statistics and JSON profile
    # -------------------------------------------------------------------------
    inhabited_n = int(inhabited.sum())
    walk_assigned_final = int(np.count_nonzero(inhabited & (walk_id != NODATA)))
    moto_assigned_final = int(np.count_nonzero(inhabited & (moto_id != NODATA)))

    # Compute summary statistics from exact values
    walk_valid_times = walk_time_out[(inhabited) & (walk_id != NODATA)]
    moto_valid_times = moto_time_out[(inhabited) & (moto_id != NODATA)]
    walk_valid_dist = walk_dist_out[(inhabited) & (walk_id != NODATA)]
    moto_valid_dist = moto_dist_out[(inhabited) & (moto_id != NODATA)]

    total_elapsed = time.perf_counter() - t_total_start

    stats = {
        "run_date": time.strftime("%Y-%m-%d %H:%M:%S"),
        "inputs": {
            "dataset_dir": str(ds),
            "population": POPULATION_TIF,
            "car_share": CAR_SHARE_TIF,
            "friction_walk": FRICTION_WALK_TIF,
            "friction_moto": FRICTION_MOTO_TIF,
            "resellers": f"{RESELLER_GPKG}:{RESELLER_LAYER}",
        },
        "parameters": {
            "beta": BETA,
            "epsilon": EPS,
            "attr_min": ATTR_MIN,
            "pixel_size_m": PIXEL_SIZE_M,
        },
        "counts": {
            "inhabited_pixels": inhabited_n,
            "seeds_walk": len(seeds_walk),
            "seeds_moto": len(seeds_moto),
            "assigned_walk": walk_assigned_final,
            "assigned_moto": moto_assigned_final,
        },
        "percentages": {
            "walk_assignment_pct": 100.0 * _safe_ratio(walk_assigned_final, inhabited_n),
            "moto_assignment_pct": 100.0 * _safe_ratio(moto_assigned_final, inhabited_n),
        },
        "timings_seconds": {
            "read_population": t_read_pop,
            "reproject": t_reproject,
            "load_seeds": t_load_seeds,
            "connected_components": t_components,
            "walk_solver_phase1": t_walk_solver,
            "moto_solver_phase1": t_moto_solver,
            "exact_recomputation": t_exact,
            "total": total_elapsed,
        },
        "outputs": {
            "raster": OUTPUT_RASTER,
            "lookup_csv": OUTPUT_LOOKUP,
        },
    }

    if OUTPUT_PROFILE:
        with open(ds / OUTPUT_PROFILE, "w", encoding="utf-8") as f:
            json.dump(stats, f, indent=2)

    # Print summary
    print("\n=== SUMMARY ===")
    print(f"Pixels processed      : {inhabited_n:,}")
    print(f"Walk assigned         : {walk_assigned_final:,} ({stats['percentages']['walk_assignment_pct']:.2f}%)")
    print(f"Moto assigned         : {moto_assigned_final:,} ({stats['percentages']['moto_assignment_pct']:.2f}%)")
    if len(walk_valid_times) > 0:
        print(f"Walk time (min)       : min={np.min(walk_valid_times):.2f} median={np.median(walk_valid_times):.2f} max={np.max(walk_valid_times):.2f}")
    if len(moto_valid_times) > 0:
        print(f"Moto time (min)       : min={np.min(moto_valid_times):.2f} median={np.median(moto_valid_times):.2f} max={np.max(moto_valid_times):.2f}")
    print(f"Total elapsed time    : {total_elapsed:.2f} seconds")
    print(f"Output raster         : {ds / OUTPUT_RASTER}")
    print(f"Lookup CSV            : {ds / OUTPUT_LOOKUP}")
    if OUTPUT_PROFILE:
        print(f"Run profile           : {ds / OUTPUT_PROFILE}")
    if len(walk_valid_dist) > 0:
        print(f"Walk distance (km)    : min={np.min(walk_valid_dist):.2f} median={np.median(walk_valid_dist):.2f} max={np.max(walk_valid_dist):.2f}")
    if len(moto_valid_dist) > 0:
        print(f"Moto distance (km)    : min={np.min(moto_valid_dist):.2f} median={np.median(moto_valid_dist):.2f} max={np.max(moto_valid_dist):.2f}")

    return stats


# -----------------------------------------------------------------------------
# Execute the pipeline
# -----------------------------------------------------------------------------
if __name__ == "__main__":
    run_stats = run_huff_pipeline()

Inhabited pixels: 563,852 | Walk seeds: 2,185 | Moto seeds: 2,185
[walk] 100,000/563,851 (17.74%) | 40718.0 px/s | ETA 0.2 min
[walk] 200,000/563,851 (35.47%) | 42168.6 px/s | ETA 0.1 min
[walk] 300,000/563,851 (53.21%) | 41022.3 px/s | ETA 0.1 min
[walk] 400,000/563,851 (70.94%) | 40805.2 px/s | ETA 0.1 min
[walk] 500,000/563,851 (88.68%) | 39801.5 px/s | ETA 0.0 min
[walk] 563,851/563,851 (100.00%) | 37289.2 px/s | ETA 0.0 min
[moto] 100,000/563,482 (17.75%) | 43483.0 px/s | ETA 0.2 min
[moto] 200,000/563,482 (35.49%) | 39223.3 px/s | ETA 0.2 min
[moto] 300,000/563,482 (53.24%) | 37472.1 px/s | ETA 0.1 min
[moto] 400,000/563,482 (70.99%) | 36077.1 px/s | ETA 0.1 min
[moto] 500,000/563,482 (88.73%) | 34348.9 px/s | ETA 0.0 min
[moto] 563,480/563,482 (100.00%) | 32365.2 px/s | ETA 0.0 min
[walk-exact] Processing 2,184 resellers...
[walk-exact] 500/2,184 resellers done (22.9%) | 45.3 res/s | ETA 0.6 min
[walk-exact] 1,000/2,184 resellers done (45.8%) | 48.6 res/s | ETA 0.4 min
[walk-exa